In [1]:
import pandas as pd
import numpy as np
from bokeh.plotting import figure, save, output_file
from bokeh.layouts import column, row
from bokeh.models import TabPanel, Tabs, Div, ColumnDataSource  # CAMBIO: TabPanel en lugar de Panel
from bokeh.palettes import Viridis256, Plasma256
import random
import math
from datetime import datetime, timedelta

# Generar datos de métricas de negocio
np.random.seed(456)

# Simular datos de ventas empresariales
fechas_ventas = pd.date_range(start='2023-01-01', end='2024-12-31', freq='D')
productos = ['Producto A', 'Producto B', 'Producto C', 'Producto D', 'Producto E']
canales = ['Online', 'Retail', 'Wholesale', 'B2B']

datos_ventas = []
for fecha in fechas_ventas:
    for producto in productos:
        for canal in canales:
            # Simular patrones estacionales
            factor_estacional = 1 + 0.3 * math.sin((fecha.dayofyear / 365) * 2 * math.pi)
            
            ventas_base = random.randint(100, 1000)
            ventas = int(ventas_base * factor_estacional * random.uniform(0.7, 1.3))
            
            datos_ventas.append({
                'fecha': fecha,
                'producto': producto,
                'canal': canal,
                'ventas': ventas,
                'ingresos': ventas * random.uniform(50, 500),
                'costo': ventas * random.uniform(20, 250)
            })

df_ventas = pd.DataFrame(datos_ventas)
df_ventas['ganancia'] = df_ventas['ingresos'] - df_ventas['costo']
df_ventas['margen_pct'] = (df_ventas['ganancia'] / df_ventas['ingresos']) * 100

def crear_dashboard_negocio():
    # TAB 1: KPI Dashboard
    def crear_kpi_dashboard():
        # Métricas agregadas
        ventas_totales = df_ventas['ventas'].sum()
        ingresos_totales = df_ventas['ingresos'].sum()
        ganancia_total = df_ventas['ganancia'].sum()
        
        # Gráfico de tendencias mensuales
        df_mensual = df_ventas.groupby(df_ventas['fecha'].dt.to_period('M')).agg({
            'ventas': 'sum',
            'ingresos': 'sum',
            'ganancia': 'sum'
        }).reset_index()
        df_mensual['fecha'] = df_mensual['fecha'].dt.to_timestamp()
        
        p1 = figure(
            title="📊 KPI Principales - Tendencias Mensuales",
            x_axis_type='datetime',
            width=1200,
            height=400,
            tools="pan,wheel_zoom,box_zoom,reset,save"
        )
        
        # Múltiples líneas para diferentes métricas
        p1.line(df_mensual['fecha'], df_mensual['ventas']/1000, 
               line_width=3, color='#3498db', legend_label='Ventas (K unidades)', alpha=0.8)
        p1.line(df_mensual['fecha'], df_mensual['ingresos']/1000000, 
               line_width=3, color='#2ecc71', legend_label='Ingresos (M USD)', alpha=0.8)
        p1.line(df_mensual['fecha'], df_mensual['ganancia']/1000000, 
               line_width=3, color='#e74c3c', legend_label='Ganancia (M USD)', alpha=0.8)
        
        p1.title.text_font_size = "16pt"
        p1.title.text_color = "#2c3e50"
        p1.legend.location = "top_left"
        p1.legend.click_policy = "hide"
        p1.background_fill_color = "#f8f9fa"
        
        # Añadir etiquetas de ejes
        p1.xaxis.axis_label = "Período"
        p1.yaxis.axis_label = "Valor Escalado"
        
        return p1
    
    # TAB 2: Análisis por Producto
    def crear_analisis_productos():
        df_productos = df_ventas.groupby('producto').agg({
            'ventas': 'sum',
            'ingresos': 'sum', 
            'ganancia': 'sum'
        }).reset_index()
        
        # Crear ColumnDataSource
        source_productos = ColumnDataSource(df_productos)
        
        p2 = figure(
            title="🛍️ Performance por Producto",
            x_range=df_productos['producto'],
            width=900,
            height=500,
            tools="hover,save"
        )
        
        # Gráfico de barras para ingresos
        p2.vbar(x='producto', top='ingresos', width=0.8, 
               source=source_productos, color='#3498db', alpha=0.8, legend_label='Ingresos')
        
        # Gráfico de barras para ganancia (más delgado)
        p2.vbar(x='producto', top='ganancia', width=0.5,
               source=source_productos, color='#2ecc71', alpha=0.8, legend_label='Ganancia')
        
        # Configurar hover
        p2.hover.tooltips = [
            ("Producto", "@producto"),
            ("Ventas", "@ventas{0,0} unidades"),
            ("Ingresos", "$@ingresos{0,0.00}"),
            ("Ganancia", "$@ganancia{0,0.00}")
        ]
        
        p2.xgrid.grid_line_color = None
        p2.legend.location = "top_right"
        p2.yaxis.axis_label = "Valor (USD)"
        p2.title.text_font_size = "14pt"
        p2.title.text_color = "#2c3e50"
        
        return p2
    
    # TAB 3: Análisis de Canales
    def crear_analisis_canales():
        df_canales = df_ventas.groupby(['fecha', 'canal']).agg({
            'ingresos': 'sum'
        }).reset_index()
        
        p3 = figure(
            title="🌐 Evolución de Ingresos por Canal de Venta",
            x_axis_type='datetime',
            width=1100,
            height=500,
            tools="pan,wheel_zoom,reset,save"
        )
        
        colores_canales = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
        
        for i, canal in enumerate(canales):
            data_canal = df_canales[df_canales['canal'] == canal]
            p3.line(data_canal['fecha'], data_canal['ingresos'],
                   line_width=3, color=colores_canales[i], 
                   legend_label=canal, alpha=0.8)
            
            # Añadir puntos para mejor visualización
            p3.scatter(data_canal['fecha'], data_canal['ingresos'],
                      size=3, color=colores_canales[i], alpha=0.6)
        
        p3.legend.location = "top_left"
        p3.legend.click_policy = "hide"
        p3.title.text_font_size = "14pt"
        p3.title.text_color = "#2c3e50"
        p3.xaxis.axis_label = "Período"
        p3.yaxis.axis_label = "Ingresos (USD)"
        
        return p3
    
    # CAMBIO: Crear tabs usando TabPanel en lugar de Panel
    tab1 = TabPanel(child=crear_kpi_dashboard(), title="📊 KPIs Principales")
    tab2 = TabPanel(child=crear_analisis_productos(), title="🛍️ Análisis Productos")  
    tab3 = TabPanel(child=crear_analisis_canales(), title="🌐 Canales de Venta")
    
    tabs = Tabs(tabs=[tab1, tab2, tab3])
    
    # Añadir header profesional
    header = Div(text="""
    <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; border-radius: 10px; margin-bottom: 20px;">
        <h1 style="text-align: center; color: white; font-family: 'Helvetica Neue', Arial, sans-serif; margin: 0; font-size: 28px;">
        📈 Dashboard Ejecutivo de Métricas de Negocio
        </h1>
        <p style="text-align: center; color: #f8f9fa; font-size: 16px; margin: 10px 0 0 0;">
        Análisis integral de performance comercial • Período 2023-2024 • Actualizado en tiempo real
        </p>
    </div>
    """)
    
    dashboard = column(header, tabs)
    return dashboard

# Crear y guardar dashboard de negocio
dashboard_negocio = crear_dashboard_negocio()
output_file("html/metricas_negocio.html")
save(dashboard_negocio)

print("✅ Dashboard de Métricas de Negocio creado: html/metricas_negocio.html")
print("📊 Dashboard con tabs interactivos listo para portafolio!")

✅ Dashboard de Métricas de Negocio creado: html/metricas_negocio.html
📊 Dashboard con tabs interactivos listo para portafolio!
